In [1]:
import numpy as np 
import pandas as pd


In [2]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder

In [8]:
df = pd.read_csv('./src/covid.csv')
df.head()

,age,gender,fever,cough,city,has_covid
0,9,Female,100,Mild,Quetta,No
1,75,Male,101,Mild,Sargodha,No
2,80,Female,98,Mild,Rawalpindi,Yes
3,39,Female,101,Mild,Karachi,Yes
4,32,Male,101,Mild,Rawalpindi,No


In [9]:
df.isnull().sum()

age          0
gender       0
fever        0
cough        0
city         0
has_covid    0
dtype: int64

In [10]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(df.drop(columns=['has_covid']),df['has_covid'],test_size=0.2)

In [11]:
X_train

,age,gender,fever,cough,city
9,55,Male,102,Mild,Multan
33,7,Male,101,Moderate,Islamabad
40,77,Female,101,Moderate,Peshawar
83,80,Female,100,Severe,Bahawalpur
60,22,Female,102,Moderate,Sialkot
...,...,...,...,...,...
31,31,Female,101,Moderate,Gujranwala
75,66,Female,102,Moderate,Lahore
94,40,Male,98,Severe,Multan
19,78,Male,98,Mild,Hyderabad


# Without Column Transformer

In [15]:
# ordinal encoding for cough
oe = OrdinalEncoder(categories=[['Mild','Moderate','Severe']])
X_train_cough = oe.fit_transform(X_train[['cough']])

X_test_cough = oe.fit_transform(X_test[['cough']])

X_train_cough.shape

(80, 1)

In [22]:
# one hot encoding on gender, city
ohe = OneHotEncoder(drop='first')
X_train_gender_city = ohe.fit_transform(X_train[['gender','city']]).toarray()

X_test_gender_city = ohe.fit_transform(X_test[['gender','city']]).toarray()
X_train_gender_city.shape

(80, 15)

In [23]:
x_train_age = X_train.drop(columns=['gender','fever','cough','city']).values
x_test_age = X_test.drop(columns=['gender','fever','cough','city']).values
x_train_age.shape

(80, 1)

In [29]:
x_train_transformed = np.concatenate((x_train_age,X_train_gender_city,X_train_cough),axis = 1)
x_test_transformed = np.concatenate((x_test_age,X_test_gender_city,X_test_cough),axis = 1)
x_train_transformed.shape

(80, 17)

# With column transformer

In [25]:
from sklearn.compose import ColumnTransformer

In [26]:
transformer = ColumnTransformer(transformers=[
    ('tnf1',OrdinalEncoder(categories=[['Mild','Moderate','Severe']]),['cough']),
    ('tnf2',OneHotEncoder(drop='first'),['gender','city'])
],remainder='passthrough')

In [32]:
transformer.fit_transform(X_train).toarray().shape

(80, 18)